# Intent Based Agentic AI Implementation

### ReActXen setup

The following cell adds the ReActXen source directory to `sys.path` and provides a small verification of core `reactxen` modules. Run the next code cell to actually perform the setup and check imports.

In [56]:
import os
import sys
from pathlib import Path

try:
    # Get the absolute path of the current notebook directory
    notebook_dir = Path(__file__).resolve().parent
    workspace_root = notebook_dir.parent
    reactxen_path = workspace_root / "ReActXen" / "src" 
    print(f"✅ Method 1 - Using __file__: {reactxen_path}")
except NameError:
    # Method 2: For Jupyter notebooks (__file__ not available)
    notebook_dir = Path.cwd()
    workspace_root = notebook_dir.parent
    reactxen_path = workspace_root / "ReActXen" / "src" 
    print(f"✅ Method 2 - Using Path.cwd(): {reactxen_path}")

# Verify the path exists
if reactxen_path.exists():
    print(f"✅ ReActXen path exists: {reactxen_path}")
    # Add to sys.path
    sys.path.insert(0, str(reactxen_path))
    print(f"✅ Added to sys.path: {reactxen_path}")
else:
    print(f"❌ ReActXen path does not exist: {reactxen_path}")
    print("Please check the directory structure.")

# Test the import
try:
    # click on imported function to check if function can be accessed
    from ReActXen.src.reactxen.prebuilt.create_reactxen_agent import create_reactxen_agent

    print("✅ Successfully imported create_reactxen_agent")
except ImportError as e:
    print(f"❌ Import failed: {e}")     # TODO : ignore this, as I just needed access to the functions internally
    # print("Available paths:")
    # for i, p in enumerate(sys.path[:5]):
    #     print(f"  {i}: {p}")

✅ Method 2 - Using Path.cwd(): /Users/ayandas/Desktop/research_ibm/Intent-Based-Industrial-Automation/ReActXen/src
✅ ReActXen path exists: /Users/ayandas/Desktop/research_ibm/Intent-Based-Industrial-Automation/ReActXen/src
✅ Added to sys.path: /Users/ayandas/Desktop/research_ibm/Intent-Based-Industrial-Automation/ReActXen/src
❌ Import failed: No module named 'ReActXen'


In [57]:
# install neccessary packages
%pip install --upgrade pip
%pip install langchain-ibm langchain-community haversine pandas langchain-ibm langchain-community haversine pandas ibm-watsonx-ai numpy xgboost scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [58]:
# Handle necessary installation and setup
# NOTE : use this code block for any/all imports related to subseuquent code logic of agentic implementation

import argparse
# from ReActXen.src.reactxen.prebuilt.create_reactxen_agent import create_reactxen_agent
import json
from getpass import getpass
import warnings
import pandas as pd
import langchain
# import xgboost as xgb   # too many configuration related errors
from langchain_ibm import WatsonxLLM
from langchain_core.prompts import PromptTemplate
from ibm_watsonx_ai import APIClient
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from getpass import getpass

# filter out all warnings
warnings.filterwarnings("ignore")

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/ayandas/.pyenv/versions/3.11.9/lib/python3.11/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <B111F8D5-6AC6-3245-A6B5-94693F6992AB> /Users/ayandas/.pyenv/versions/3.11.9/lib/python3.11/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/

In [60]:
!python -c "import sys; print(f'{sys.maxsize:x}')"      # test to see why xgboost is breaking

7fffffffffffffff


In [52]:
'''
define 
'''


Name: scikit-learn
Version: 1.7.2
Summary: A set of python modules for machine learning and data mining
Home-page: https://scikit-learn.org
Author: 
Author-email: 
License-Expression: BSD-3-Clause
Location: /Users/ayandas/.pyenv/versions/3.11.9/lib/python3.11/site-packages
Requires: joblib, numpy, scipy, threadpoolctl
Required-by: 


In [ ]:
# TODO : implement the logic behind data pre-processing, which can then be used for the ML models upon preparation
# TODO : wrap them around reusable functions, which can then be used as a tool instead

In [39]:
'''define additional parameters that create_reactxen_agent will accept'''

agent_prompt = PromptTemplate(
    input_variables=["train_data","test_data","ground_truth"],
    template=(
        "You are an AI Assistant specialized in predictive, corrective, preventative and reliabillity centered maintenance",
        "\nYou have been given three sets of datasets and the breakdown of each of them is as follows:\n\n"
        "{train_data} : Contains complete run-to-failure trajectories for 100 engines. Each row represents one time step (cycle) of an engine. Each engine runs from a healthy state (cycle 1) to its simulated failure cycle\n"
        "{test_data} : Contains partial trajectories involving another 100 engines, with each engine stopping right before failure. Which means the full lifetime cannot be seen. This data can be used to predict how many more remaining useful lifecycles (RUL for short) remains\n"
        
        # This might not be neccessary
        # "{ground_truth} : Contains 100 numbers, one per test engine, and contains the true number of remaining useful cycles after the last record of the engine's trajectory. This is the 'answer' key to be used for evaluation"
    )
)


# There are 2 methods involved when it comes to calculating RUL
#
# This is something to improve the thinking process of agents
# 1. Linear RUL
# 2. Piecewise RUL
reflect_prompt = PromptTemplate(
    input_variables=["predicted_val", "actual_val"],
    template=(
        # This part of the prompt will be useful for prediction of rul
        # This is a binary scenario
        "In the event that {predicted_val} != {actual_val} using the tools you have been provided, reflect on why this occured and provide a single liner justification. Then try again, don't just look at the final answer, understand in-depth the steps needed to be taken to arrive at the final answer."
        "In the event that {predicted_val} == {actual_val} using the tools you have been provided, provide a brief explanation of which machine learning technique was used to derive at this answer, and whether you have tested with other models as well. The explanation should be factual, and should match a typical SME knowledge."
        "In addition to that, consider which amongst the list of prediction techniques has proven to yield the highest accuracy of prediction that matches the actual value"
        "In the event that the problem is more open-ended, perform in-depth root cause analysis and derive a simple explanation justifying the response that has been provided, reference the neccessary data/facts as part of the justification."
    ),
)

# additional parameters that will be included within the model

# TODO : implement input parsing logic
actionstyle = "code" | "text"
kwargs = {
    "actionstyle" : actionstyle,
    "max_steps" : 6,
    "num_reflect_iteration" : 3,
    "early_stop" : False,
    "debug" : False,
    
    # TODO : implement the reactstyle involved, choose 1, right now both options has been included
    "reactstyle" : "thought_and_act_together" | "only_act"
    
}

Absolute Batman


In [54]:
# potential utterance we are working with
sample_utterace = "We should focus on the engines that are running on fumes, which ones are likely to give out in the next 20 cycles? Provide me a list of engine ids."

list_of_model_ids = [
    "ibm/granite-3-8b-instruct",
    "ibm/granite-13b-instruct-v2",
    "meta-llama/llama-4-maverick-17b-128e-instruct-fp8",
]

In [ ]:
# load and store the values involved on the environmental variable file


@misc{Sahoo_Data-Driven_Remaining_Useful_2020,
author = {Sahoo, Biswajit},
doi = {10.5281/zenodo.5890595},
month = {9},
title = {Data-Driven Remaining Useful Life (RUL) Prediction},
url = {https://biswajitsahoo1111.github.io/rul_codes_open/},
year = {2020}
}

In [44]:
reactxen_agent = create_reactxen_agent(question=sample_utterace, key=, react_llm_model=list_of_model_ids[0], reflect_llm_model_id=list_of_model_ids[0])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 16.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 29.4 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [6]:
# TODO : list of tools to implement and how agent should interact
'''
an agent that has access to this info (this will be a subcomponent of overall flow)

1. get_reference_train_data
2. get_reference_test_data
3. get_rul_model
4. train_rul_model
5. predict_rul_model
'''

In [ ]:
def get_reference_train_data():
    '''fairly straightforward, retrieves the train data and for mats it using pandas'''
    pass

def get_reference_test_data():
    '''fairly straightforward, retrieves the test data and formats it using pandas'''
    pass

def get_rul_model():
    '''
    simply a getter function (might be a bit redundant unless series of training and testing model logic is present)
    
    Go to model registry/catalog, can be queried to get rul models, and get 2 pipelines, model registry is key-value pair
    
    based on rul, use the 2 option available to select list 
    1. linear piecewise
    2. linear rul
    
    non-deterministic approach, LLM makes sense of things based on the information based on what info I have, what model I have.
    
    consider searching in internet to identify any better solution that may have come, in a current setting, the solution is fixed, but new models may come in market --> might build a solution that is dynamic, it varies with the time, rather than static solution.
    
    It needs to adapt, every week, it's going out and bringing new model, it means model registry increased.
    
    Can ask LLM these are several models, and can ask to suggest several different models
    
    Another way, have most popular model, and use a leaderboard, which is downloaded or one with the best performance in a certain leaderboard.
    
    most recent doesn't gurantee it's performant --> this will keep changing --> do one thing --> tool for model selection (decision tree based to help the LLM select the model), bring LLM to help choose the model
    '''
    pass

def train_rul_model():
    '''
    sub-agent that can call on the functions responsible for running the training on the models
    
    smallest_subproblem : include the logic for training of at least one technique
    
    retrieve the machine learning model to the pre-processed data and train on it
    
    ideal appraoch : model selection using LLM, this way making solution dynamic, tommorow if I change data, LLM will help me select the right model based on that knowledge, now this becomes agentic, LLM is making certain decision.
    
    Many options, and use LLM to choose the right option based on the data.
    '''
    pass

# sub-agent that will have access to various methods of prediction via tool
# NOTE : double check to determine whether or not this is required.
def predict_rul_agent():
    '''
    It's tools will involve using the functions for predicting RUL based on the models that has been trained
    
    smallest_subproblem : include the logic for prediction of at least one techniques
    '''
    pass

'''
think about why we need an agent to solve this problem when a software engineer can solve?
'''